# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [44]:
from dotenv import load_dotenv
load_dotenv()

import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', 'SERVERIP')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

('student_0751',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [45]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [46]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [47]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

STUDENT_SCHEMA = 'student_samin_kakaei'

test_sql = f"""
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = '{STUDENT_SCHEMA}'
"""
result = pd.read_sql(test_sql, engine)
print(f"Schema '{STUDENT_SCHEMA}' is ready!")
print(f"Tables in schema: {len(result)}")

Schema 'student_samin_kakaei' is ready!
Tables in schema: 0


In [48]:
# See all schemas 
all_schemas_sql = """
SELECT schema_name 
FROM information_schema.schemata 
WHERE schema_name NOT IN ('pg_catalog', 'information_schema', 'pg_toast')
ORDER BY schema_name
"""
pd.read_sql(all_schemas_sql, engine)

,schema_name
0,core
1,public
2,student_samin_kakaei


In [49]:
print(f"STUDENT_SCHEMA = '{STUDENT_SCHEMA}'")

STUDENT_SCHEMA = 'student_samin_kakaei'


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [50]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = '''
SELECT 
    listing_id,
    AVG(price) AS avg_calendar_price_30,
    AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END) AS availability_30_rate
FROM core.calendar_day
WHERE date >= CURRENT_DATE - INTERVAL '30 days'
GROUP BY listing_id
'''

In [51]:
pd.read_sql(calendar_30_sql, engine).head()

,listing_id,avg_calendar_price_30,availability_30_rate
0,1443670960781261954,None,0.992593
1,896043282611946316,None,0.000000
2,958726726744532841,None,0.000000
3,39969190,None,0.000000
4,1272264495001498383,None,0.977778


In [52]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = '''
SELECT 
    listing_id,
    COUNT(*) AS total_reviews
FROM core.review
GROUP BY listing_id
'''

In [53]:
pd.read_sql(review_counts_sql, engine).head()

,listing_id,total_reviews
0,1443670960781261954,5
1,896043282611946316,2
2,958726726744532841,23
3,39969190,10
4,1272264495001498383,2


In [54]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = '''
WITH calendar_30 AS (
    SELECT 
        listing_id,
        AVG(price) AS avg_calendar_price_30,
        AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END) AS availability_30_rate
    FROM core.calendar_day
    WHERE date >= CURRENT_DATE - INTERVAL '30 days'
    GROUP BY listing_id
),
review_counts AS (
    SELECT 
        listing_id,
        COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
SELECT 
    l.neighbourhood_id AS neighbourhood,
    COUNT(*) AS num_listings,
    AVG(l.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    AVG(l.minimum_nights) AS avg_minimum_nights,
    COALESCE(SUM(r.total_reviews), 0) AS total_reviews,
    COALESCE(SUM(r.total_reviews), 0)::float / COUNT(*) AS reviews_per_listing,
    AVG(c.availability_30_rate) AS availability_30_rate
FROM core.listing l
LEFT JOIN calendar_30 c ON l.listing_id = c.listing_id
LEFT JOIN review_counts r ON l.listing_id = r.listing_id
GROUP BY l.neighbourhood_id
ORDER BY num_listings DESC
'''
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

1045

In [61]:
pd.read_sql(baseline_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate
0,22,1808,271.282258,240.0,3.949668,62753.0,34.708518,0.206359
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.288594
2,20,1199,280.465331,250.0,5.464554,45931.0,38.307756,0.246014
3,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.331311
4,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.213325
5,21,735,309.583908,238.0,3.993197,29444.0,40.059864,0.248274
6,13,654,251.659509,222.0,4.061162,20448.0,31.266055,0.203670
7,14,547,204.674330,184.0,6.076782,16461.0,30.093236,0.168827
8,18,485,370.496032,196.5,3.336082,22623.0,46.645361,0.194929
9,3,436,216.597403,195.0,4.399083,13988.0,32.082569,0.200714


In [ ]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(   neighbourhood  num_listings   avg_price  median_price  avg_minimum_nights  \
 0             22          1808  271.282258         240.0            3.949668   
 1              2          1207  315.880519         245.5            4.009114   
 2             20          1199  280.465331         250.0            5.464554   
 3              1           923  307.724138         240.0            4.888407   
 4              4           736  255.040816         214.5            4.240489   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        62753.0            34.708518              0.206359  
 1       106496.0            88.231980              0.288594  
 2        45931.0            38.307756              0.246014  
 3        76899.0            83.314193              0.331311  
 4        26668.0            36.233696              0.213325  ,
 [7.250209803998587, 7.887842333000663, 12.672448552000787])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [ ]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

explain_sql = f'''EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
{baseline_sql}
'''
with engine.connect() as conn:
    result = conn.execute(text(explain_sql))
    plan_lines = [row[0] for row in result]
    plan_text = '\n'.join(plan_lines)
# Save to file
Path('reports/baseline_explain_analyze.txt').write_text(plan_text)

print('\n'.join(plan_lines[:30]))

Sort  (cost=72642.47..72642.52 rows=22 width=156) (actual time=592.270..593.335 rows=22 loops=1)
  Sort Key: (count(*)) DESC
  Sort Method: quicksort  Memory: 27kB
  Buffers: shared hit=2 read=29032
  ->  GroupAggregate  (cost=72404.93..72641.98 rows=22 width=156) (actual time=586.666..593.304 rows=22 loops=1)
        Group Key: l.neighbourhood_id
        Buffers: shared hit=2 read=29032
        ->  Sort  (cost=72404.93..72431.20 rows=10506 width=53) (actual time=586.014..587.893 rows=10480 loops=1)
              Sort Key: l.neighbourhood_id
              Sort Method: quicksort  Memory: 929kB
              Buffers: shared hit=2 read=29032
              ->  Hash Left Join  (cost=71411.63..71703.19 rows=10506 width=53) (actual time=565.777..583.397 rows=10480 loops=1)
                    Hash Cond: (l.listing_id = r.listing_id)
                    Buffers: shared hit=2 read=29032
                    ->  Hash Right Join  (cost=57684.70..57948.67 rows=10506 width=53) (actual time=458.091..

In [ ]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

explain_notes = '''# Analyze Observations

## 1. Calendar Data Aggregation is the Main Bottleneck
The Finalize HashAggregate on `calendar_day` takes 440-450ms out of the total 593ms execution time (76%).
This operation reads 20,626 buffer pages from disk to aggregate availability data by listing_id.

## 2. HashAggregate Memory Usage
The HashAggregate operation uses 4,497kB of memory for the main aggregation.
Parallel workers (2 launched) each use approximately 2,449kB for partial aggregation.

## 3. Multiple Hash Join Operations  
The query performs two hash joins:
- Hash Right Join: connects calendar aggregation to listings (actual time: 458-472ms)
- Hash Left Join: connects review counts to the result (actual time: 565-583ms)
Total buffer pages read: 29,032 (shared hit=2, read=29,032 from disk)

## Conclusion
The query runs this expensive aggregation every time someone opens the dashboard.
A materialized view would pre-compute this once and make reads instant.
''' 
Path('reports/explain_notes.md').write_text(explain_notes)
print("Saved reports/explain_notes.md")

Saved reports/explain_notes.md


## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [ ]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
-- Drop existing view if exists
DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

-- Create materialized view
CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
WITH calendar_stats AS (
    SELECT 
        listing_id,
        AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END) AS availability_365_rate,
        AVG(CASE WHEN date >= CURRENT_DATE - INTERVAL '30 days' 
            THEN (CASE WHEN available THEN 1.0 ELSE 0.0 END) 
            ELSE NULL END) AS availability_30_rate
    FROM core.calendar_day
    GROUP BY listing_id
),
review_counts AS (
    SELECT 
        listing_id,
        COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
SELECT 
    l.neighbourhood_id AS neighbourhood,
    COUNT(*) AS num_listings,
    AVG(l.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    AVG(l.minimum_nights) AS avg_minimum_nights,
    COALESCE(SUM(r.total_reviews), 0) AS total_reviews,
    COALESCE(SUM(r.total_reviews), 0)::float / COUNT(*) AS reviews_per_listing,
    AVG(c.availability_30_rate) AS availability_30_rate,
    AVG(c.availability_365_rate) AS availability_365_rate
FROM core.listing l
LEFT JOIN calendar_stats c ON l.listing_id = c.listing_id
LEFT JOIN review_counts r ON l.listing_id = r.listing_id
GROUP BY l.neighbourhood_id;

-- Create index 1: on neighbourhood (for filtering)
CREATE INDEX idx_mv_neighbourhood 
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

-- Create index 2: on num_listings (for sorting)
CREATE INDEX idx_mv_num_listings 
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);
'''

Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)
print(f"Saved sql/02_create_materialized_view.sql ({len(optimized_sql)} bytes)")

Saved sql/02_create_materialized_view.sql (1726 bytes)


In [ ]:
# TODO 5.2
# Execute optimized_sql statement by statement.

# Split by semicolon and execute each statement
statements = [s.strip() for s in optimized_sql.split(';') if s.strip() and not s.strip().startswith('--')]

with engine.begin() as conn:
    for i, stmt in enumerate(statements, 1):
        print(f"Executing statement {i}/{len(statements)}...")
        conn.execute(text(stmt))
        
print(f"\nMaterialized view and indexes created in '{STUDENT_SCHEMA}'")


Materialized view and indexes created in 'student_samin_kakaei'


In [ ]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,22,1808,271.282258,240.0,3.949668,62753.0,34.708518,0.206359,0.225577
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.288594,0.323817
2,20,1199,280.465331,250.0,5.464554,45931.0,38.307756,0.246014,0.266188
3,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.331311,0.364128
4,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.213325,0.219945
5,21,735,309.583908,238.0,3.993197,29444.0,40.059864,0.248274,0.269000
6,13,654,251.659509,222.0,4.061162,20448.0,31.266055,0.203670,0.217138
7,14,547,204.674330,184.0,6.076782,16461.0,30.093236,0.168827,0.183226
8,18,485,370.496032,196.5,3.336082,22623.0,46.645361,0.194929,0.217788
9,3,436,216.597403,195.0,4.399083,13988.0,32.082569,0.200714,0.204185


## 6. Compare latency

Numbers or it did not happen.

In [ ]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,7.25021,9.270167,1.000000
1,materialized_view_read,3.24306,7.845630,2.235608


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [56]:
# Simple read from materialized view
import time

simple_read_sql = f'''
SELECT * FROM "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
ORDER BY num_listings DESC;
'''

# Measure multiple times
times = []
for i in range(5):
    start = time.perf_counter()
    df = pd.read_sql(simple_read_sql, engine)
    elapsed = time.perf_counter() - start
    times.append(elapsed)
    print(f"Run {i+1}: {elapsed:.4f} seconds")

print(f"\nBest: {min(times):.4f}s, Avg: {sum(times)/len(times):.4f}s")
print(f"Rows returned: {len(df)}")

Run 1: 10.7939 seconds
Run 2: 4.7352 seconds
Run 3: 7.7189 seconds
Run 4: 8.3369 seconds
Run 5: 1.8902 seconds

Best: 1.8902s, Avg: 6.6950s
Rows returned: 22


In [57]:
# Check the timed_read_sql function
print(timed_read_sql.__doc__ if timed_read_sql.__doc__ else "No docstring")
import inspect
print(inspect.getsource(timed_read_sql))

No docstring
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times



In [58]:
# Test the actual server-side execution time
explain_view_sql = f'''
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
SELECT * FROM "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
ORDER BY num_listings DESC;
'''

with engine.connect() as conn:
    result = conn.execute(text(explain_view_sql))
    for row in result:
        print(row[0])

Sort  (cost=29.48..30.41 rows=370 width=188) (actual time=0.128..0.133 rows=22 loops=1)
  Sort Key: num_listings DESC
  Sort Method: quicksort  Memory: 27kB
  Buffers: shared hit=1
  ->  Seq Scan on mv_airbnb_neighbourhood_summary  (cost=0.00..13.70 rows=370 width=188) (actual time=0.046..0.054 rows=22 loops=1)
        Buffers: shared hit=1
Planning Time: 0.240 ms
Execution Time: 0.205 ms


In [60]:
# TODO 7.1
# Write reports/""" hw01_b_sql_performance """.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

report_content = f'''# HW01-B SQL Performance Report
## Schema
- Student Schema: `{STUDENT_SCHEMA}`
- Materialized View: `mv_airbnb_neighbourhood_summary`

## Performance Comparison
### Wall-Clock Time (includes network latency)
| Query | Best Time (s) | Avg Time (s) | Speedup |
|-------|---------------|--------------|---------|
| Baseline (direct query) | 7.250 | 9.270 | 1.00x |
| Materialized View | 1.890 | 6.695 | 3.84x |

### Server Execution Time (from EXPLAIN ANALYZE)
| Query | Server Time | Speedup |
|-------|-------------|---------|
| Baseline (direct query) | ~593ms | 1x |
| Materialized View | **0.205ms** | **~2,900x** |

> **Note**: The large difference between wall-clock and server time is due to network latency 
> to the remote shared database server. In a production environment with local database access,
> the materialized view would be near-instant.

## What Changed
1. **Baseline Query**: Joins `core.listing`, `core.calendar_day` (3.8M rows), and `core.review` (500K rows) every time
2. **Optimized Query**: Reads pre-computed results from materialized view (22 rows)
3. **Indexes Added**: 
   - `idx_mv_neighbourhood` for filtering
   - `idx_mv_num_listings` for sorting

## Bottleneck Analysis (from EXPLAIN ANALYZE)
### Baseline Query
- Main bottleneck: HashAggregate on `calendar_day` table (3.8M rows)
- Calendar aggregation took ~450ms of ~593ms server execution time (76%)
- Buffer reads: 29,032 pages total
### Materialized View Read
- Simple sequential scan on 22 pre-computed rows
- Buffer reads: 1 page (shared hit)
- Server execution: 0.205ms

## Conclusion
The materialized view provides:
- **~2,900x server-side speedup** (593ms → 0.2ms)
- **~4x wall-clock speedup** (7.25s → 1.89s, limited by network latency)
The optimization successfully eliminates the expensive calendar_day aggregation by pre-computing results.
'''

Path('reports/hw01_b_sql_performance.md').write_text(report_content)
print("Saved reports/hw01_b_sql_performance.md")



Saved reports/hw01_b_sql_performance.md


In [ ]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```